# Day 2 Runner — Data Pipeline + Continued Pretraining (CPT)

Turn **Qwen3-1.7B-Base** into a domain-adapted base (`cpt-v1`) for an LLM-tutor.
Full fine-tuning on an A40 (~44 GB). Runs on a rented GPU (RunPod, etc.).

Pipeline: **collect → clean → split → tokenize/pack → CPT → perplexity.**
All settings live in `configs/day2.yaml`. Edit there, not in the cells.

> **Tested-on-RunPod notes are baked in:** the install cell handles the blinker /
> transformers / torchaudio issues, and every script call uses `{sys.executable}`
> so it runs under the same Python as this kernel.

## 0. GPU + code

In [ ]:
!nvidia-smi

In [1]:
# Get the repo — pick ONE.
# Option A: clone
# !git clone https://github.com/vinmlops/llm-from-base-to-assistant.git
# %cd llm-from-base-to-assistant
# Option B: unzip an uploaded archive
#!unzip -q llm-from-base-to-assistant.zip 
%cd llm-from-base-to-assistant
import os; print(os.getcwd())

/workspace/llm-from-base-to-assistant
/workspace/llm-from-base-to-assistant


## 1. Install dependencies (text-only CPT, RunPod-safe)

This notebook does **full continued pretraining**, so **PEFT/LoRA is not required on Day 2**.
The previous PEFT import was the source of the `BloomPreTrainedModel` failure, so this
setup deliberately removes PEFT (and unused compiled audio/vision packages) from this environment.

The install cell:
- installs the repo requirements first;
- pins the Hugging Face training stack used by this notebook;
- leaves the existing CUDA-enabled PyTorch build untouched;
- removes `peft`, `torchvision`, and `torchaudio` because Day 2 does not use them;
- avoids `--no-deps` and `--force-reinstall`, which often create inconsistent environments.

**After running the install cell, restart the kernel/runtime once, then run the verification cell.**


In [ ]:
import importlib.util

print(importlib.util.find_spec("peft"))

In [ ]:
!grep -RniE "PeftModel|LoraConfig|get_peft_model|import peft|from peft" \
    training/ data/ evaluation/ configs/ || true

In [ ]:
import shutil
from pathlib import Path

p = Path("/usr/local/lib/python3.12/dist-packages/peft")

if p.exists():
    shutil.rmtree(p)
    print("Removed stale PEFT directory:", p)
else:
    print("No stale PEFT directory found")

In [ ]:
import sys
import subprocess

PY = sys.executable

# Remove packages not needed for full CPT and any broken PEFT installation
subprocess.run(
    [PY, "-m", "pip", "uninstall", "-y",
     "peft", "trl", "torchvision", "torchaudio"],
    check=False
)

# Do NOT reinstall torch — keep RunPod's CUDA-enabled torch.
subprocess.check_call([
    PY, "-m", "pip", "install",
    "--upgrade",
    "--no-cache-dir",
    "transformers==4.57.6",
    "tokenizers==0.22.2",
    "accelerate==1.15.0",
    "datasets==5.0.1",
    "sentencepiece>=0.2.0",
    "pyyaml>=6.0.2",
    "requests>=2.32.0",
    "beautifulsoup4>=4.12.3",
    "huggingface-hub>=0.34.0"
])

print("\nINSTALL COMPLETE")
print("NOW RESTART THE JUPYTER KERNEL BEFORE RUNNING ANY OTHER CELL.")

In [ ]:
import sys, subprocess

PY = sys.executable

def pip(*args):
    cmd = [PY, '-m', 'pip', *args]
    print('>', ' '.join(cmd))
    subprocess.check_call(cmd)

# Keep pip tooling healthy.
pip('install', '-U', 'pip', 'setuptools', 'wheel')

# Install project-specific packages first.
pip('install', '-r', 'requirements.txt', '--ignore-installed', 'blinker')

# Day 2 is FULL CPT, not LoRA/PEFT. Remove packages that caused the
# BloomPreTrainedModel / compiled-extension import failures.
subprocess.run([PY, '-m', 'pip', 'uninstall', '-y', 'peft', 'torchvision', 'torchaudio'], check=False)

# Pin only the Hugging Face stack. Do NOT reinstall torch; keep the
# CUDA build already supplied by the GPU image. Let pip choose the
# compatible tokenizers/huggingface-hub versions automatically.
pip('install', '--upgrade', '--no-cache-dir',
    'transformers==4.57.6',
    'accelerate==1.15.0',
    'datasets==5.0.1')

print('\nInstall finished.')
print('IMPORTANT: restart the kernel/runtime NOW before importing transformers.')


In [2]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

In [3]:
import sys
import importlib.util
import importlib.metadata as md

import torch
import transformers
import accelerate
import datasets

print("Python      :", sys.executable)
print("torch       :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("accelerate  :", accelerate.__version__)
print("datasets    :", datasets.__version__)

print("\nPEFT module spec:", importlib.util.find_spec("peft"))

try:
    print("PEFT package:", md.version("peft"))
except md.PackageNotFoundError:
    print("PEFT package: NOT INSTALLED — correct for Day 2")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)

print("\nCore training imports: OK")

from transformers.models.qwen3.configuration_qwen3 import Qwen3Config
print("Qwen3 support: OK")


Python      : /usr/local/bin/python
torch       : 2.8.0+cu128
CUDA        : True
transformers: 4.57.6
accelerate  : 1.15.0
datasets    : 5.0.1

PEFT module spec: None
PEFT package: NOT INSTALLED — correct for Day 2

Core training imports: OK
Qwen3 support: OK


## 2. Collect raw text
Papers (`ai-arxiv`) + scraped HF/PyTorch docs + theory (HF blog, HF LLM Course,
d2l, arXiv surveys) + FineWeb-Edu replay → `data/raw/`.
Flags: `--skip-docs`, `--skip-theory` if a source is flaky.

In [ ]:
import sys
!{sys.executable} data/collect.py --config configs/day2.yaml

## 3. Clean + deduplicate
Then **read ~20 cleaned docs by hand** — catches more than any metric.

In [4]:
import sys
!{sys.executable} data/clean.py --config configs/day2.yaml

[clean] domain_papers.jsonl: kept 400, dropped_short 0, dropped_dup 0
[clean] domain_docs.jsonl: kept 147, dropped_short 1, dropped_dup 2
[clean] replay.jsonl: kept 800, dropped_short 0, dropped_dup 0

Tip: open a few files in data/clean/ and read some docs by hand before continuing.
Next: python split.py --config day2_cpt.yaml


In [5]:
from itertools import islice
import json
with open("data/clean/domain_papers.jsonl") as f:
    for line in islice(f, 2):
        r = json.loads(line); print(r["source"], "|", r["text"][:400], "\n---")

papers | UNDERSTANDING HTML WITH LARGE LANGUAGE
MODELS
Izzeddin Gur, Oﬁr Nachum, Yingjie Miao, Mustafa Safdari, Austin Huang
Aakanksha Chowdhery, Sharan Narang, Noah Fiedel, Aleksandra Faust
Google Research
fizzeddin,ofirnachum,yingjiemiao,msafdari,austinvhuang
chowdhery,sharannarang,nfiedel,sandrafaust g@google.com
ABSTRACT
Large language models (LLMs) have shown exceptional performance on a va-
riety of  
---
papers | Published as a conference paper at ICLR 2019
DECOUPLED WEIGHT DECAY REGULARIZATION
Ilya Loshchilov & Frank Hutter
University of Freiburg
Freiburg, Germany,
filya,fhg@cs.uni-freiburg.de
ABSTRACT
L2regularization and weight decay regularization are equivalent for standard
stochastic gradient descent (when rescaled by the learning rate), but as we demon-
strate this is notthe case for adaptive gradie 
---


## 4. Split (document-level) + leakage check
Look for "✓ no leakage found".

In [6]:
import sys
!{sys.executable} data/split.py --config configs/day2.yaml

[split] domain=547 train_domain=466 replay=800 val=27 heldout=54
Next: python tokenize_pack.py --config day2_cpt.yaml


## 5. Tokenize + pack
1024-token blocks, 85/15 domain:replay mixture, token-budget capped.

In [7]:
import sys
!{sys.executable} data/tokenize_pack.py --config configs/day2.yaml

tokenizer_config.json: 9.68kB [00:00, 29.8MB/s]
vocab.json: 2.78MB [00:00, 51.2MB/s]
merges.txt: 1.67MB [00:00, 52.4MB/s]
tokenizer.json: 7.03MB [00:00, 88.9MB/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (220962 > 131072). Running this sequence through the model will result in indexing errors
Saving the dataset (1/1 shards): 100%|█| 2931/2931 [00:00<00:00, 382177.54 exam
Saving the dataset (1/1 shards): 100%|█| 293/293 [00:00<00:00, 147054.10 exampl
[pack] train_blocks=2931 train_tokens=3001344 (domain=2550784, replay=450560, ~85% domain)
[pack] val_blocks=293 val_tokens=300032
Next: python cpt.py --config day2_cpt.yaml


## 6. Continued pretraining (full fine-tuning)
Watch the loss: smooth descent = healthy; spikes = LR too high; rising **val** loss
= overfitting/forgetting. Saves `artifacts/cpt-v1`. (~15-60 min on the A40.)

> `cpt.py` is version-robust — it adapts its TrainingArguments to your installed
> transformers, so the 4.x/5.x API differences won't crash it.

### CPT preflight

Day 2 performs **full fine-tuning**, so `training/cpt.py` should not import PEFT/LoRA.
The check below catches an accidental PEFT dependency before a long GPU run.


In [ ]:
from pathlib import Path
cpt_source = Path('training/cpt.py').read_text(encoding='utf-8')
bad = [s for s in ('import peft', 'from peft', 'LoraConfig', 'get_peft_model') if s in cpt_source]
if bad:
    raise RuntimeError('training/cpt.py unexpectedly depends on PEFT/LoRA: ' + ', '.join(bad) +
                       '. Day 2 is configured for full CPT; remove that PEFT import/use first.')
print('training/cpt.py PEFT check: OK (full CPT)')


In [8]:
import sys
!{sys.executable} training/cpt.py --config configs/day2.yaml

Loading base model: Qwen/Qwen3-1.7B-Base
config.json: 100%|████████████████████████████| 727/727 [00:00<00:00, 4.54MB/s]
Tokenizer: eos=151643 pad=151643 | model vocab_size=151936
train blocks: 2931 | val blocks: 293
OK: every block is exactly 1024 tokens
train stats: {'tokens': 3001344, 'eos_count': 625, 'min_token_id': 0, 'max_token_id': 151643}
val stats  : {'tokens': 300032, 'eos_count': 16, 'min_token_id': 0, 'max_token_id': 151643}

--- sample of packed text ---
[block 2619] ' strategies evolve alongside the expansion of cities.\n"It is crucial to discover how birds adapt to transformations in their habitat so that we can decrease their effects," said Ibanez-Alamo..\n"Predation change caused by city growth is serious," outlined Ibanez-Alamo.\nAs the scientist indicates, tactics against their hunters are "crucial" so that birds can adapt to their new environment: "Birds sh'

[block 456] 'ronym: Seq2Seq\nTitle: Densely Connected Convolutional Networks for Image Classification\nAcro

## 7. Perplexity — did CPT work?
**Want:** domain perplexity ↓, general perplexity ≈ flat.

In [18]:
import sys
!{sys.executable} evaluation/perplexity.py --config configs/day2.yaml

Scoring BASE ...
`torch_dtype` is deprecated! Use `dtype` instead!
Scoring CPT ...
The tokenizer you are loading from 'artifacts/cpt-v1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.

=== Perplexity (lower = better) ===
model          domain    general
BASE             7.54      10.62
CPT              6.76      11.13

Want: domain DOWN (specialized), general ~flat (didn't forget).
domain change : -10.3%
general change: +4.8%

Saved results to artifacts/cpt-v1/perplexity.json


In [19]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("artifacts/cpt-v1")
# round-trip check
text = "The capital of France is Paris."
ids = tok.encode(text)
back = tok.decode(ids)
print(ids)
print(back)
print(back.strip() == text or text in back)

The tokenizer you are loading from 'artifacts/cpt-v1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


[785, 6722, 315, 9625, 374, 12095, 13]
The capital of France is Paris.
True
